In [3]:
import os
import numpy as np
import evaluate
import shutil
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from IPython.display import FileLink

# 1. Load and Tokenize
dataset = load_dataset("ag_news")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(example):
    return tokenizer(example["text"], padding="max_length", truncation=True, max_length=128)

tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset.set_format("torch")

# 2. Model & Metrics
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=4)
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy.compute(predictions=predictions, references=labels)["accuracy"]
    f1_score = f1.compute(predictions=predictions, references=labels, average="weighted")["f1"]
    return {"accuracy": acc, "f1": f1_score}

# 3. Optimized Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    fp16=True,                       # Fast training on P100
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=1,              # Set to 1 for a quick first test use 3 for final
    eval_strategy="no",              # Skip eval during training to save time
    save_strategy="no",
    report_to="none"
)

# 4. Train
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics
)

print("Starting training...")
trainer.train()

# 5. Save and Zip
model_path = "./news_topic_model"
trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)

shutil.make_archive('news_topic_model', 'zip', model_path)
print("Training complete and file zipped!")

# 6. Create Download Link
FileLink('news_topic_model.zip')

ModuleNotFoundError: No module named 'evaluate'

ERROR: Could not find a version that satisfies the requirement evalulate (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for evalulate
